# Processing

This notebook processes the external data and produces a datased valid to produce plots that answers those questions

- Open papers vs closed papers by year (has catalan science moved to a open access or closed one?)
- Price comparision of top 5 papers on each science domain per year (should see that the covid year price is lower for health science?)
- Quantity of catalan published papers on top 5 publishers (are we depending on specific editorials)? Contrast with european journals.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA = Path('../data')
FIGS = Path('../plots/adri')
FIGS.mkdir(exist_ok=True)
EXTERNAL = DATA/'external'
PROCESSED = DATA/'processed'

In [3]:
# ---------- 1. Cargar ----------
works  = pd.read_parquet(EXTERNAL/'works.parquet')
open_access  = pd.read_parquet(EXTERNAL/'work_open_access.parquet')
papers = pd.read_parquet(EXTERNAL/'papers_cleaned.gencat.parquet')
hosts  = pd.read_parquet(EXTERNAL/'host_orgs.parquet')
fields = pd.read_parquet(EXTERNAL/'fields.parquet')
sources = pd.read_parquet(EXTERNAL/'work_sources.parquet')

In [4]:
papers_only_dois_and_dates = papers[['doi', 'date']]

papers_only_dois_and_dates['year'] = pd.to_datetime(papers_only_dois_and_dates['date']).dt.year
papers_only_dois_and_dates = papers_only_dois_and_dates.drop(columns='date').drop_duplicates(ignore_index=True)
papers_only_dois_and_dates['doi'] = papers_only_dois_and_dates['doi'].str.strip()
papers_only_dois_and_dates

,doi,year
0,10.1210/jcem-9-9-883,1949
1,10.1002/cber.19520850704,1952
2,10.1111/j.1365-2133.1952.tb15900.x,1952
3,10.1038/jid.1953.55,1953
4,10.1016/s0022-5347(17)67795-4,1954
...,...,...
367616,10.21203/rs.3.rs-957988/v1,2022
367617,10.1016/j.cmpb.2022.107314,2022
367618,10.1344/anpsic2022.52/2.3,2022
367619,10.1016/j.jsat.2021.108496,2022


In [5]:
joined = pd.merge(left=works, right=open_access, left_on='openalex_id', right_on='work_openalex_id')
joined = pd.merge(left=joined, right=fields, left_on='primary_topic_field_id', right_on='field_id')
joined = pd.merge(left=joined, right=sources, left_on='openalex_id', right_on='work_openalex_id')
joined = pd.merge(left=joined, right=hosts, left_on='openalex_id', right_on='openalex_id')
joined['doi'] = joined['doi'].str.strip()
joined = pd.merge(left=joined, right=papers_only_dois_and_dates, left_on='doi', right_on='doi')
joined = joined[joined['source_type'] == 'journal']

joined = joined.drop(columns=['work_openalex_id_x', 'work_openalex_id_y', 'field_id', 'doi', 'awards', 'funders', 'referenced_works', 'topics_field_ids', 'primary_topic_field_id', 'field_name', 'source_type'])

RENAME = {'host_organization_id'  : 'publisher_id',
          'host_organization_name': 'publisher_name', 'source_id'             : 'journal_id', 'source_display_name'   : 'journal_name'
          }

joined = joined.rename(columns=RENAME)
joined = joined.reset_index(drop=True)
joined.columns

Index(['openalex_id', 'apc_paid_value_usd', 'apc_list_value_usd',
       'is_open_access', 'open_access_status', 'domain_id', 'domain_name',
       'journal_id', 'journal_name', 'publisher_id', 'publisher_name', 'year'],
      dtype='str')

In [7]:
joined.to_parquet(PROCESSED/'apc_oa_works.parquet')